### Inital look and saving cusips to CSV

In [1]:
import duckdb
import pandas as pd
db_path = r"C:\Users\agand\Desktop\Personal_Coding\Finance_Projects\MultiFactorModel_LFQA\multi-factor-models\venv\crsp\crsp_dataset.duckdb"
parquet_file = r"C:\Users\agand\Desktop\Personal_Coding\Finance_Projects\MultiFactorModel_LFQA\multi-factor-models\venv\crsp\CRSPdF20022025.parquet"

conn = duckdb.connect(db_path)

conn.execute(f"""
CREATE TABLE IF NOT EXISTS crsp_table AS
SELECT * FROM read_parquet('{parquet_file}')
""")

df_preview = conn.execute("SELECT * FROM crsp_table LIMIT 5").fetchdf()
df_preview


,cusip,date,permno,permco,cfacpr,cfacshr,shrout,prc,ret,retx,vol,ask,askhi,bid,bidlo
0,36720410,2002-01-02,10001,7953,1.5,1.5,2566.0,11.10,-0.030568,-0.030568,6603.0,11.35,11.49,11.19,11.1000
1,05978R10,2002-01-02,10002,7954,1.0,1.0,8530.0,9.60,-0.029323,-0.029323,55000.0,9.89,9.60,9.27,9.6000
2,23326910,2002-01-02,10012,7969,1.0,1.0,20924.0,3.25,0.041667,0.041667,50975.0,3.25,3.29,3.24,3.0600
3,44950710,2002-01-02,10019,7971,1.0,1.0,8282.0,1.05,-0.160000,-0.160000,53250.0,1.05,1.30,1.03,1.0497
4,00103110,2002-01-02,10025,7975,1.0,1.0,7794.0,23.80,-0.006678,-0.006678,7030.0,23.90,24.00,23.80,23.4000


In [3]:
# Get all unique CUSIPs
cusip_df = conn.execute("""
SELECT DISTINCT cusip 
FROM crsp_table
ORDER BY cusip
""").fetchdf()

print(f"Number of unique CUSIPs: {len(cusip_df)}")
cusip_df



Number of unique CUSIPs: 20553


,cusip
0,00030710
1,00032Q10
2,00036020
3,00036110
4,00037520
...,...
20548,Y9233510
20549,Y9369110
20550,Y9384M10
20551,Y9390M10


In [2]:
#cusip_df.to_csv("cusips.csv")
conn.close()

### Beginning to load factors and work on data

In [19]:
import pandas as pd
import duckdb
import statsmodels.api as sm
import numpy as np

#### Cleaning and alinging the data

In [4]:
import duckdb

db_path = r"C:\Users\agand\Desktop\Personal_Coding\Finance_Projects\MultiFactorModel_LFQA\multi-factor-models\venv\crsp\crsp_dataset.duckdb"
parquet_file = r"C:\Users\agand\Desktop\Personal_Coding\Finance_Projects\MultiFactorModel_LFQA\multi-factor-models\venv\crsp\CRSPdF20022025.parquet"

con = duckdb.connect(db_path)

In [ ]:
con.execute(f"""
CREATE TABLE IF NOT EXISTS crsp_table AS
SELECT * FROM read_parquet('{parquet_file}');
""")

In [12]:
con.execute("""
DROP TABLE IF EXISTS crsp_clean;
CREATE TABLE crsp_clean AS
SELECT
  cusip,
  CAST(date AS DATE) AS date,
  permno,
  permco,
  cfacpr,
  cfacshr,
  CASE WHEN shrout IS NULL OR shrout <= 0 THEN NULL ELSE shrout END AS shrout,
  CASE WHEN prc    IS NULL OR prc    <= 0 THEN NULL ELSE prc    END AS prc,
  ret,
  retx,
  CASE WHEN vol    IS NULL OR vol    <= 0 THEN NULL ELSE vol    END AS vol,
  CASE WHEN ask    IS NULL OR ask    <= 0 THEN NULL ELSE ask    END AS ask,
  askhi,
  CASE WHEN bid    IS NULL OR bid    <= 0 THEN NULL ELSE bid    END AS bid,
  bidlo
FROM crsp_table
WHERE cusip IS NOT NULL AND date IS NOT NULL
ORDER BY cusip, date;
""")

In [13]:
summary = con.execute("""
SELECT 
  COUNT(*) AS total_rows,
  COUNT(DISTINCT cusip) AS unique_cusips,
  MIN(date) AS min_date,
  MAX(date) AS max_date,
  SUM(prc    IS NULL)::BIGINT AS null_prc,
  SUM(shrout IS NULL)::BIGINT AS null_shrout,
  SUM(vol    IS NULL)::BIGINT AS null_vol,
  SUM(ask    IS NULL)::BIGINT AS null_ask,
  SUM(bid    IS NULL)::BIGINT AS null_bid
FROM crsp_clean;
""").fetchdf()
print("Summary: ")
print(summary)

dupes = con.execute("""
SELECT cusip, date, COUNT(*) AS n
FROM crsp_clean
GROUP BY 1,2
HAVING COUNT(*) > 1
LIMIT 25;
""").fetchdf()
print("Sample duplicates per (cusip, date):")
print(dupes)

Summary: 
   total_rows  unique_cusips   min_date   max_date  null_prc  null_shrout  \
0    40802455          20553 2002-01-02 2023-12-29   1758743            1   

   null_vol  null_ask  null_bid  
0   1566793    565855    565815  
Sample duplicates per (cusip, date):
Empty DataFrame
Columns: [cusip, date, n]
Index: []


In [14]:
con.execute("""
DROP TABLE IF EXISTS crsp_gaps;
CREATE TABLE crsp_gaps AS
WITH seq AS (
  SELECT
    cusip,
    date,
    LAG(date) OVER (PARTITION BY cusip ORDER BY date) AS prev_date
  FROM crsp_clean
)
SELECT
  cusip,
  date,
  prev_date,
  CASE
    WHEN prev_date IS NULL THEN NULL
    ELSE DATE_DIFF('day', prev_date, date)
  END AS gap_days
FROM seq
WHERE prev_date IS NOT NULL;
""")


In [15]:
con.execute("""
DROP TABLE IF EXISTS crsp_continuity;
CREATE TABLE crsp_continuity AS
SELECT
  c.cusip,
  MIN(c.date) AS start_date,
  MAX(c.date) AS end_date,
  COUNT(*) AS obs,
  COUNT(DISTINCT c.date) AS distinct_dates,
  SUM(CASE WHEN g.gap_days IS NOT NULL THEN 1 ELSE 0 END) AS transitions,
  SUM(CASE WHEN g.gap_days > 7 THEN 1 ELSE 0 END) AS gaps_gt_7d,
  MAX(COALESCE(g.gap_days,0)) AS max_gap_days,
  AVG(CASE WHEN g.gap_days IS NOT NULL THEN g.gap_days END) AS avg_gap_days
FROM crsp_clean c
LEFT JOIN crsp_gaps g
  ON c.cusip = g.cusip AND c.date = g.date
GROUP BY 1;
""")

In [16]:
continuity_flag = con.execute("""
SELECT *
FROM crsp_continuity
ORDER BY gaps_gt_7d DESC, max_gap_days DESC
LIMIT 50;
""").fetchdf()
print("Worst continuity cases:")
print(continuity_flag)

Worst continuity cases:
       cusip start_date   end_date   obs  distinct_dates  transitions  \
0   78470V10 2011-07-27 2020-01-14  2131            2131       2130.0   
1   22087310 2002-01-02 2009-09-14  1939            1939       1938.0   
2   44894750 2002-01-02 2023-12-29  5537            5537       5536.0   
3   46138E53 2007-10-11 2023-12-29  4083            4083       4082.0   
4   16938G10 2007-10-05 2017-05-16  2420            2420       2419.0   
5   49918380 2003-12-18 2012-07-16  2159            2159       2158.0   
6   59132A10 2010-04-09 2013-04-12   758             758        757.0   
7   29384810 2002-01-02 2009-07-28  1906            1906       1905.0   
8   36146210 2002-01-02 2008-04-30  1593            1593       1592.0   
9   24610V10 2002-01-02 2022-02-11  5065            5065       5064.0   
10  78464A70 2002-01-02 2023-12-29  5537            5537       5536.0   
11  45661Q10 2005-06-21 2010-11-05  1356            1356       1355.0   
12  73935X55 2006-09-20 200

In [17]:
con.execute("""
CREATE OR REPLACE VIEW v_crsp_base AS
SELECT
  cusip, date, permno, permco,
  prc, shrout, vol, ask, bid, askhi, bidlo, ret, retx,
  /* pre-compute useful primitives; results will be NULL if inputs are NULL */
  (prc * shrout * 1000) AS market_cap,
  CASE WHEN shrout IS NULL OR vol IS NULL THEN NULL ELSE vol / shrout END AS turnover,
  CASE 
    WHEN ask IS NULL OR bid IS NULL OR (ask + bid) = 0 THEN NULL
    ELSE (ask - bid) / ((ask + bid) / 2.0)
  END AS bid_ask_spread
FROM crsp_clean;
""")


In [20]:
print("Views and tables ready: crsp_table, crsp_clean, crsp_gaps, crsp_continuity, v_crsp_base")
con.execute("SHOW TABLES;").fetchdf()


Views and tables ready: crsp_table, crsp_clean, crsp_gaps, crsp_continuity, v_crsp_base


,name
0,crsp_clean
1,crsp_continuity
2,crsp_gaps
3,crsp_table
4,v_crsp_base


#### Making Base Variables

In [21]:
con.execute("""
DROP TABLE IF EXISTS factors_daily;
CREATE TABLE factors_daily AS
SELECT
  c.cusip,
  c.date,
  c.permno,
  c.permco,
  c.prc,
  c.shrout,
  c.vol,
  c.ask,
  c.bid,
  c.askhi,
  c.bidlo,
  c.ret,
  c.retx,
  /* Market cap: prc * shrout * 1000 (shrout in thousands) */
  CASE WHEN c.prc IS NULL OR c.shrout IS NULL
       THEN NULL
       ELSE c.prc * c.shrout * 1000
  END AS market_cap,
  /* Turnover: vol / shrout */
  CASE WHEN c.vol IS NULL OR c.shrout IS NULL OR c.shrout = 0
       THEN NULL
       ELSE c.vol / c.shrout
  END AS turnover,
  /* Bid-ask spread: (ask - bid) / ((ask + bid)/2) */
  CASE 
    WHEN c.ask IS NULL OR c.bid IS NULL OR (c.ask + c.bid) = 0
    THEN NULL
    ELSE (c.ask - c.bid) / ((c.ask + c.bid) / 2.0)
  END AS bid_ask_spread
FROM crsp_clean c
ORDER BY c.cusip, c.date;
""")


Monthly aggregation
- market_cap_month_end: value at month-end (arg_max by date)
- turnover_month_avg: simple monthly average of daily turnover
- spread_month_avg: simple monthly average of daily spread

In [22]:

con.execute("""
DROP TABLE IF EXISTS factors_monthly;
CREATE TABLE factors_monthly AS
WITH tagged AS (
  SELECT
    cusip,
    date,
    date_trunc('month', date) AS month,
    market_cap,
    turnover,
    bid_ask_spread
  FROM factors_daily
)
SELECT
  cusip,
  month AS month_start,
  /* month_end for reference */
  (DATE_TRUNC('month', month) + INTERVAL 1 MONTH - INTERVAL 1 DAY)::DATE AS month_end,
  /* market cap at month-end (last available day in month) */
  ARG_MAX(market_cap, date) AS market_cap_month_end,
  /* monthly averages (NULLs ignored) */
  AVG(turnover) AS turnover_month_avg,
  AVG(bid_ask_spread) AS spread_month_avg,
  COUNT(*) AS trading_days_in_month
FROM tagged
GROUP BY cusip, month
ORDER BY cusip, month;
""")

Quick sanity pulls

In [23]:
print(con.execute("""
SELECT COUNT(*) AS daily_rows, COUNT(DISTINCT cusip) AS cusips
FROM factors_daily;
""").fetchdf())

print(con.execute("""
SELECT COUNT(*) AS monthly_rows, MIN(month_start) AS min_m, MAX(month_start) AS max_m
FROM factors_monthly;
""").fetchdf())

print("Built: factors_daily, factors_monthly.")


   daily_rows  cusips
0    40802455   20553
   monthly_rows      min_m      max_m
0       1956558 2002-01-01 2023-12-01
Built: factors_daily, factors_monthly.


In [4]:
con.execute("SHOW TABLES;").fetch_df()

,name
0,crsp_clean
1,crsp_continuity
2,crsp_gaps
3,crsp_table
4,factors_daily
5,factors_monthly
6,v_crsp_base


#### Computing rolling factors

Monthly total returns from daily 'ret' (compound within month)

In [5]:
con.execute("""
DROP TABLE IF EXISTS returns_monthly;
CREATE TABLE returns_monthly AS
WITH m AS (
  SELECT
    cusip,
    DATE_TRUNC('month', date)::DATE AS month_start,
    /* guard against invalid log when ret <= -1 or NULL */
    EXP(SUM(LOG(1 + ret)) FILTER (WHERE ret IS NOT NULL AND ret > -0.999999)) - 1 AS ret_m
  FROM crsp_clean
  GROUP BY 1,2
)
SELECT * FROM m
ORDER BY cusip, month_start;
""")

Join monthly bases (size, turnover, spread) with monthly returns

In [6]:
con.execute("""
DROP TABLE IF EXISTS monthly_base;
CREATE TABLE monthly_base AS
SELECT
  fm.cusip,
  fm.month_start,
  fm.month_end,
  fm.market_cap_month_end,
  fm.turnover_month_avg,
  fm.spread_month_avg,
  rm.ret_m
FROM factors_monthly fm
LEFT JOIN returns_monthly rm
  USING (cusip, month_start)
ORDER BY cusip, month_start;
""")

Momentum (21-1): compound t-12... t-2, require full 11 months

In [7]:
con.execute("""
DROP TABLE IF EXISTS monthly_with_mom;
CREATE TABLE monthly_with_mom AS
WITH w AS (
  SELECT
    cusip,
    month_start,
    market_cap_month_end,
    turnover_month_avg,
    spread_month_avg,
    ret_m,
    /* window counts and sums over the 11-month window excluding t-1 */
    COUNT(ret_m) OVER (
      PARTITION BY cusip
      ORDER BY month_start
      ROWS BETWEEN 12 PRECEDING AND 2 PRECEDING
    ) AS win_count,
    SUM(LOG(1 + ret_m)) OVER (
      PARTITION BY cusip
      ORDER BY month_start
      ROWS BETWEEN 12 PRECEDING AND 2 PRECEDING
    ) AS win_logsum
  FROM monthly_base
)
SELECT
  cusip,
  month_start,
  market_cap_month_end,
  turnover_month_avg,
  spread_month_avg,
  ret_m,
  /* require exactly 11 non-null months to form MOM(12-1) */
  CASE WHEN win_count = 11 THEN EXP(win_logsum) - 1 ELSE NULL END AS mom_12_1
FROM w
ORDER BY cusip, month_start;
""")


Build raw factor exposures, then z-score within month

In [8]:
con.execute("""
DROP TABLE IF EXISTS factor_exposures_z;
CREATE TABLE factor_exposures_z AS
WITH raw AS (
  SELECT
    cusip,
    month_start,
    /* size: log of market cap to reduce skew */
    CASE WHEN market_cap_month_end IS NULL OR market_cap_month_end <= 0
         THEN NULL ELSE LOG(market_cap_month_end) END AS size_raw,
    turnover_month_avg       AS turn_raw,
    spread_month_avg         AS bas_raw,
    mom_12_1                 AS mom_raw
  FROM monthly_with_mom
),
z AS (
  SELECT
    cusip,
    month_start,
    /* z = (x - mean)/std within each month; handle zero or NULL stddev */
    (size_raw - AVG(size_raw) OVER (PARTITION BY month_start))
      / NULLIF(STDDEV_SAMP(size_raw) OVER (PARTITION BY month_start), 0) AS size_z,
    (mom_raw - AVG(mom_raw) OVER (PARTITION BY month_start))
      / NULLIF(STDDEV_SAMP(mom_raw) OVER (PARTITION BY month_start), 0) AS mom_z,
    (turn_raw - AVG(turn_raw) OVER (PARTITION BY month_start))
      / NULLIF(STDDEV_SAMP(turn_raw) OVER (PARTITION BY month_start), 0) AS turnover_z,
    (bas_raw - AVG(bas_raw) OVER (PARTITION BY month_start))
      / NULLIF(STDDEV_SAMP(bas_raw) OVER (PARTITION BY month_start), 0) AS baspread_z
  FROM raw
)
SELECT * FROM z
ORDER BY cusip, month_start;
""")


Persist a clean exposures table with requested columns

In [9]:
con.execute("""
DROP TABLE IF EXISTS factor_exposures_monthly;
CREATE TABLE factor_exposures_monthly AS
SELECT
  cusip,
  month_start AS date,
  size_z      AS size,
  mom_z       AS mom,
  turnover_z  AS turnover,
  baspread_z  AS ba_spread
FROM factor_exposures_z
ORDER BY cusip, date;
""")


Sanity Check 

In [17]:
con.execute("""
SELECT 
  COUNT(*) AS rows,
  MIN(date) AS min_month,
  MAX(date) AS max_month,
  SUM(size IS NULL)::BIGINT AS null_size_z,
  SUM(mom IS NULL)::BIGINT AS null_mom_z,
  SUM(turnover IS NULL)::BIGINT AS null_turnover_z,
  SUM(ba_spread IS NULL)::BIGINT AS null_ba_spread_z
FROM factor_exposures_monthly;
""").fetchdf()

,rows,min_month,max_month,null_size_z,null_mom_z,null_turnover_z,null_ba_spread_z
0,1956558,2002-01-01,2023-12-01,25385,264717,25173,24680


In [12]:
con.execute("""
SELECT * FROM factor_exposures_monthly
ORDER BY date DESC, cusip
LIMIT 10;
""").fetchdf()

,cusip,date,size,mom,turnover,ba_spread
0,00032Q10,2023-12-01,-0.681859,-1.573853,0.013764,-0.158987
1,00036020,2023-12-01,1.181806,0.428530,-0.107662,-0.476781
2,00036110,2023-12-01,0.794619,0.840095,-0.089296,-0.498470
3,00038020,2023-12-01,1.122281,1.072141,0.003912,-0.507194
4,00039J10,2023-12-01,0.303706,0.452862,-0.085829,-0.509361
5,00039J20,2023-12-01,0.060483,0.377775,-0.065075,-0.489553
6,00039J30,2023-12-01,-1.094719,NaN,-0.102054,-0.462245
7,00039J40,2023-12-01,-1.380825,NaN,-0.113984,-0.457327
8,00039J50,2023-12-01,-0.134565,NaN,-0.107021,-0.456097
9,00039J60,2023-12-01,-0.478843,NaN,-0.092370,-0.320670


Built: returns_monthly, monthly_base, monthly_with_mom, factor_exposures_monthly

In [15]:
con.execute("SHOW TABLES;").fetch_df()

,name
0,crsp_clean
1,crsp_continuity
2,crsp_gaps
3,crsp_table
4,factor_exposures_monthly
5,factor_exposures_z
6,factors_daily
7,factors_monthly
8,monthly_base
9,monthly_with_mom


### Quick Summary of what has been done so far

So far I have computed all factors on a monthly frequency to reduce noise standardize everything. After I loaded CRSP data into duckdb and cleaning it into crsp_clean table. Continuity tables, "crsp_gaps" and "crsp_continuity" were made to check for gaps in each cusips trading history. from the cleaned data some daily base variables were calculated, market-cap, turnover, and bid-ask spread, and then stored in "factors_daily". They were then aggregated into monthly and now are in the "factors_monthly" table. Monthly returns were found from daily returns, in "returns monthly" table, and combined with factors into "Monthly base" table to caluclate momentum (12-1) by compounding months t-12 through t-2 ("monthly_with_mom" table). Each factor was then standardized within each month using z scores ("factor_exposures_z" table) and the final analysis-ready data is in the "factor_exposures_monthly" data table. it contains one record per cusip per month with standardized exposure for regression and port. analysis

Summary of tables (chatgpt)

| **Table Name**               | **What It Contains (Content)**                                                                                                    | **Why It Exists (Purpose)**                                                                                     |
| ---------------------------- | --------------------------------------------------------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------------------------- |
| **crsp_table**               | Raw CRSP parquet data loaded directly from file (`cusip`, `date`, `prc`, `ret`, `vol`, etc.)                                      | Acts as the original unmodified dataset so you always have a baseline reference.                                |
| **crsp_clean**               | Same as `crsp_table` but with `date` cast to `DATE` and all invalid numeric values (≤0) converted to `NULL`.                      | Provides a clean, analysis-ready version. Keeps all rows while ensuring invalid data doesn’t bias calculations. |
| **crsp_gaps**                | For each `(cusip, date)`, the difference in days from the previous trading date (`gap_days`).                                     | Used to check trading-date continuity and identify missing stretches in each security’s history.                |
| **crsp_continuity**          | One row per CUSIP summarizing continuity stats: start/end date, obs count, gap counts, max/avg gap.                               | Gives diagnostics on data completeness per stock before factor computation.                                     |
| **v_crsp_base** *(view)*     | View on `crsp_clean` adding computed columns: `market_cap`, `turnover`, and `bid_ask_spread`.                                     | Quick reusable base for later queries so you don’t have to rewrite those formulas.                              |
| **factors_daily**            | Daily per-CUSIP values of `market_cap`, `turnover`, and `bid_ask_spread`.                                                         | Step 1 of factor construction. Forms daily building blocks that can be aggregated to monthly.                   |
| **factors_monthly**          | Monthly aggregates: month-end market cap, average turnover, average bid-ask spread.                                               | Step 2 aggregation to monthly frequency (standard for factor models).                                           |
| **returns_monthly**          | Monthly compounded total return from daily returns.                                                                               | Needed to calculate the momentum factor and for regressions on returns later.                                   |
| **monthly_base**             | Join of `factors_monthly` + `returns_monthly` → one record per CUSIP/month with all key vars.                                     | Combines financial factors and returns into a single monthly dataset.                                           |
| **monthly_with_mom**         | Extends `monthly_base` by adding the 12-1 momentum (`mom_12_1`) computed over rolling 12 months excluding the most recent.        | Step 3—adds the time-series-based momentum signal to each stock.                                                |
| **factor_exposures_z**       | Same data as `monthly_with_mom` but with normalized (z-scored) versions of each factor: size, momentum, turnover, bid-ask spread. | Step 4—standardization within each month so all factors are comparable in regressions or portfolio weights.     |
| **factor_exposures_monthly** | Final clean output: per-month per-CUSIP standardized factor exposures (`size`, `mom`, `turnover`, `ba_spread`).                   | The final dataset for cross-sectional regressions, backtests, and portfolio construction.                       |


### Getting factor returns

In [ ]:
df = con.execute("""
SELECT 
    e.cusip AS cusip,
    e.date  AS month,
    e.size,
    e.mom,
    e.turnover,
    e.ba_spread,
    r.ret_m
FROM factor_exposures_monthly AS e
JOIN returns_monthly AS r
  ON e.cusip = r.cusip AND e.date = r.month_start
ORDER BY e.date, e.cusip;
""").fetchdf()

factor_returns = []
min_obs = 30  
for month, g in df.groupby("month"):
    g = g.replace([np.inf, -np.inf], np.nan)
    g = g.dropna(subset=["ret_m", "size", "mom", "turnover", "ba_spread"])
    if len(g) < min_obs:
        continue

    Xcols = []
    for col in ["size", "mom", "turnover", "ba_spread"]:
        if g[col].std(skipna=True) and g[col].std(skipna=True) > 0:
            Xcols.append(col)

    if not Xcols:
        continue  

    X = g[Xcols]
    y = g["ret_m"].astype(float)

    X = sm.add_constant(X, has_constant="add")
    try:
        model = sm.OLS(y, X, missing='drop').fit()
    except Exception:
        continue 

    row = {"month": month, "const": model.params.get("const", np.nan),
           "size": np.nan, "mom": np.nan, "turnover": np.nan, "ba_spread": np.nan}
    for k in Xcols:
        row[k] = model.params.get(k, np.nan)
    factor_returns.append(row)

factor_ret_df = pd.DataFrame(factor_returns).sort_values("month")

con.register("factor_ret_df", factor_ret_df)
con.execute("DROP TABLE IF EXISTS factor_returns_monthly;")
con.execute("""
CREATE TABLE factor_returns_monthly AS
SELECT * FROM factor_ret_df
ORDER BY month;
""")

print(factor_ret_df.tail())

         month     const      size       mom  turnover  ba_spread
247 2023-08-01 -0.026566  0.006245  0.014252 -0.002216  -0.003370
248 2023-09-01 -0.026555  0.003107  0.009891  0.001593  -0.001611
249 2023-10-01 -0.029952  0.004056  0.012483  0.006758  -0.006589
250 2023-11-01  0.028471  0.009600  0.004848  0.008253  -0.003821
251 2023-12-01  0.029859  0.010691 -0.002810  0.000864  -0.003999


In [ ]:
factor_ret_df.tail()

NameError: name 'factor_ret_df' is not defined

### Getting portfolios